In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/annotated/checkpoints/pre_cell_2.pickle

trying: ['pd']
me:  0
trying: ['customer']
me:  1
trying: ['load_customer']
me:  1
trying: ['tpch_parent']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2
trying: ['load_orders']
me:  2
Declaring variable pd
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable customer
Declaring variable load_customer
Declaring variable orders
Declaring variable load_orders


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Build distribution of counts for customers with ≥1 order
dist = (
    cust_order_counts
    .value_counts()
    .rename("CUSTDIST")
    .reset_index()
)
dist.columns = ["C_COUNT", "CUSTDIST"]

# compute how many customers had zero orders
zeros = len(customer) - len(cust_order_counts)
if zeros:
    # use concat instead of append (unsupported in cudf.DataFrame)
    zero_df = pd.DataFrame({"C_COUNT": [0], "CUSTDIST": [zeros]})
    dist = pd.concat([dist, zero_df], ignore_index=True)

# 3) Sort and assign to total
total = dist.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])

CPU times: user 63.2 ms, sys: 23.9 ms, total: 87 ms
Wall time: 86.8 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q13/rewritten/o4_mini_high/checkpoints/post_cell_2_try_4.pickle

migration speed (bps): 753490120.6533661
---------------------------
variables to migrate:
STORAGE_OPTS 64
dist 48
zero_df 48
total 48
customer 48
zeros 28
pd 72
load_customer 144
orders 48
tpch_parent 54
cust_order_counts 48
load_orders 144
DATA_ROOT 80
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q13/rewritten/o4_mini_high/checkpoints/post_cell_2_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['customer']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['customer']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['dist', 'total', 'zero_df', 'zeros', 'cust_order_counts']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q13/opt_cell_exec_info_2_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/annotated/checkpoints/post_cell_2.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
